# Data Analysis

## Setup

In [1]:
# Install packages
from google.colab import drive
import pandas as pd
import duckdb
import numpy as np
import os

# Connect to Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
claims = pd.read_csv("/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/Cleaned/claims_clean.csv")
members = pd.read_csv("/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/Cleaned/members_clean.csv")
providers = pd.read_csv("/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/Cleaned/providers_clean.csv")

print("Claims:", claims.shape)
print("Members:", members.shape)
print("Providers:", providers.shape)

Claims: (332740, 12)
Members: (50000, 8)
Providers: (2500, 6)


In [3]:
# Register pandas df for querying by duckdb
con = duckdb.connect()

con.register("claims", claims)
con.register("members", members)
con.register("providers", providers)

## TLDR Summary

Outpatient Procedure is the largest allowed cost category, accounting for 27% of total costs. Spending is driven primarily by high procedure costs rather than high utilization, with four procedures accounting for 86% of outpatient costs. Out-of-network services for these procedures cost about 20% more than comparable in-network services, representing a total modeled cost reduction opportunity of $787K. For next quarter, a 25% shift in out-of-network utilization toward in-network care is recommended, representing approximately $197K in modeled savings.

#### Full Analysis Summary

* Main cost area: Outpatient procedures make up 27% of total costs.
* Main cause: High cost per procedure, not high utilization.
* Top procedures: Four procedures make up 86% of outpatient costs.
* Providers: There isn't enough data per provider to identify individual high-cost providers.
* Network: Out-of-network procedures cost about 20% more.
* Geography: State differences are smaller, generally 5-10%.
* Members: High-cost members tend to receive more expensive procedures, not just more procedures.
* Opportunity: Moving all current out-of-network utilization toward observed in-network rates represents about 787K dollars in modeled cost reduction opportunity.
* **Recommendation:** Target a 25% reduction in out-of-network utilization for the four highest-cost procedures next quarter.
* Expected impact: A 25% shift represents approximately 197K dollars in modeled savings.
* Measurement: Track out-of-network utilization rate, out-of-network claim lines, average allowed cost per line, total allowed cost, and estimated savings.

## Analysis

In [ ]:
baseline = con.execute("""
SELECT
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT claim_id) AS claims,
    COUNT(DISTINCT member_id) AS members,
    COUNT(DISTINCT provider_id) AS providers,
    ROUND(SUM(allowed_amount), 2) AS total_allowed_amount,
    MIN(service_date) AS earliest_service_date,
    MAX(service_date) AS latest_service_date
FROM claims
""").df()

baseline

,claim_lines,claims,members,providers,total_allowed_amount,earliest_service_date,latest_service_date
0,332740,207653,37652,2500,87789069.21,2021-01-03,2025-12-31


In [ ]:
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332740 entries, 0 to 332739
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   claim_id               332740 non-null  object 
 1   service_line_number    332740 non-null  int64  
 2   member_id              332740 non-null  object 
 3   provider_id            332740 non-null  object 
 4   service_date           332740 non-null  object 
 5   procedure_code         332740 non-null  object 
 6   procedure_description  332740 non-null  object 
 7   diagnosis_code         332740 non-null  object 
 8   diagnosis_description  332740 non-null  object 
 9   place_of_service       332740 non-null  object 
 10  service_category       332740 non-null  object 
 11  allowed_amount         332740 non-null  float64
dtypes: float64(1), int64(1), object(10)
memory usage: 30.5+ MB


In [ ]:
claims.columns.tolist()

['claim_id',
 'service_line_number',
 'member_id',
 'provider_id',
 'service_date',
 'procedure_code',
 'procedure_description',
 'diagnosis_code',
 'diagnosis_description',
 'place_of_service',
 'service_category',
 'allowed_amount']

In [ ]:
# Summarize allowed cost and utilization by service category to identify the highest cost category.

category_summary = con.execute("""
WITH category_cost AS (
    SELECT
        service_category,
        COUNT(*) AS claim_lines,
        COUNT(DISTINCT claim_id) AS claims,
        COUNT(DISTINCT member_id) AS members,
        SUM(allowed_amount) AS total_allowed_amount,
        AVG(allowed_amount) AS avg_allowed_per_line
    FROM claims
    GROUP BY service_category
),

overall_cost AS (
    SELECT
        SUM(allowed_amount) AS overall_allowed_amount
    FROM claims
)

SELECT
    c.service_category,
    c.claim_lines,
    c.claims,
    c.members,
    ROUND(c.total_allowed_amount, 2) AS total_allowed_amount,
    ROUND(c.avg_allowed_per_line, 2) AS avg_allowed_per_line,
    ROUND(
        100.0 * c.total_allowed_amount / o.overall_allowed_amount,
        2
    ) AS pct_total_allowed_cost
FROM category_cost c
CROSS JOIN overall_cost o
ORDER BY c.total_allowed_amount DESC
""").df()

category_summary

,service_category,claim_lines,claims,members,total_allowed_amount,avg_allowed_per_line,pct_total_allowed_cost
0,Outpatient Procedure,15063,13926,10932,23678838.10,1571.99,26.97
1,Primary Care,96430,81447,30673,14477060.24,150.13,16.49
2,Emergency,26779,24926,16646,13867736.55,517.86,15.80
3,Imaging,33301,30786,19138,12598127.47,378.31,14.35
4,Specialist Visit,59965,52169,25503,11156860.68,186.06,12.71
5,Urgent Care,31262,27871,17973,4645144.69,148.59,5.29
6,Inpatient,6641,6480,5729,2498914.42,376.29,2.85
7,Preventive Care,10098,9500,8012,1707821.29,169.12,1.95
8,Therapy,14837,13596,10728,1638821.64,110.46,1.87
9,Laboratory,38364,34821,20649,1519744.13,39.61,1.73


In [ ]:
# Compare utilization and cost intensity across service categories to determine what is driving total allowed cost.

cost_decomposition = con.execute("""
SELECT
    service_category,
    COUNT(*) AS claim_lines,
    ROUND(SUM(allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_claim_lines,
    ROUND(
        100.0 * SUM(allowed_amount) / SUM(SUM(allowed_amount)) OVER (),
        2
    ) AS pct_allowed_cost
FROM claims
GROUP BY service_category
ORDER BY total_allowed_amount DESC
""").df()

cost_decomposition

,service_category,claim_lines,total_allowed_amount,avg_allowed_per_line,pct_claim_lines,pct_allowed_cost
0,Outpatient Procedure,15063,23678838.10,1571.99,4.53,26.97
1,Primary Care,96430,14477060.24,150.13,28.98,16.49
2,Emergency,26779,13867736.55,517.86,8.05,15.80
3,Imaging,33301,12598127.47,378.31,10.01,14.35
4,Specialist Visit,59965,11156860.68,186.06,18.02,12.71
5,Urgent Care,31262,4645144.69,148.59,9.40,5.29
6,Inpatient,6641,2498914.42,376.29,2.00,2.85
7,Preventive Care,10098,1707821.29,169.12,3.03,1.95
8,Therapy,14837,1638821.64,110.46,4.46,1.87
9,Laboratory,38364,1519744.13,39.61,11.53,1.73


In [ ]:
# Measure outpatient procedure utilization per member to determine whether repeat utilization is a major cost driver.

outpatient_utilization = con.execute("""
WITH member_utilization AS (
    SELECT
        member_id,
        COUNT(*) AS procedure_lines,
        SUM(allowed_amount) AS member_allowed_amount
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
    GROUP BY member_id
)

SELECT
    COUNT(*) AS members,
    SUM(procedure_lines) AS claim_lines,
    ROUND(AVG(procedure_lines), 2) AS avg_lines_per_member,
    MEDIAN(procedure_lines) AS median_lines_per_member,
    MAX(procedure_lines) AS max_lines_per_member,
    ROUND(AVG(member_allowed_amount), 2) AS avg_allowed_per_member,
    ROUND(MEDIAN(member_allowed_amount), 2) AS median_allowed_per_member
FROM member_utilization
""").df()

outpatient_utilization

,members,claim_lines,avg_lines_per_member,median_lines_per_member,max_lines_per_member,avg_allowed_per_member,median_allowed_per_member
0,10932,15063.0,1.38,1.0,8,2166.01,1020.36


In [ ]:
# Identify which procedures within Outpatient Procedure contribute the most to total allowed cost.

outpatient_procedures = con.execute("""
WITH procedure_summary AS (
    SELECT
        procedure_code,
        procedure_description,
        COUNT(*) AS claim_lines,
        COUNT(DISTINCT member_id) AS members,
        SUM(allowed_amount) AS total_allowed_amount,
        AVG(allowed_amount) AS avg_allowed_per_line
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
    GROUP BY
        procedure_code,
        procedure_description
),

outpatient_total AS (
    SELECT
        SUM(allowed_amount) AS total_outpatient_allowed
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
)

SELECT
    p.procedure_code,
    p.procedure_description,
    p.claim_lines,
    p.members,
    ROUND(p.total_allowed_amount, 2) AS total_allowed_amount,
    ROUND(p.avg_allowed_per_line, 2) AS avg_allowed_per_line,
    ROUND(
        100.0 * p.total_allowed_amount / o.total_outpatient_allowed,
        2
    ) AS pct_outpatient_cost
FROM procedure_summary p
CROSS JOIN outpatient_total o
ORDER BY p.total_allowed_amount DESC
""").df()

outpatient_procedures

,procedure_code,procedure_description,claim_lines,members,total_allowed_amount,avg_allowed_per_line,pct_outpatient_cost
0,47562,Laparoscopic cholecystectomy,1418,1365,7060138.20,4978.94,29.82
1,29881,Knee arthroscopy with meniscectomy,1456,1399,4853707.90,3333.59,20.50
2,49591,"Repair of anterior abdominal hernia, initial, ...",1538,1483,4799370.60,3120.53,20.27
3,66984,Cataract removal with intraocular lens insertion,1463,1415,3643712.20,2490.58,15.39
4,45378,Diagnostic colonoscopy,1529,1473,1351430.49,883.87,5.71
5,43235,Diagnostic upper gastrointestinal endoscopy,1548,1493,1128174.00,728.79,4.76
6,11102,"Tangential skin biopsy, single lesion",1526,1460,254062.20,166.49,1.07
7,20610,Major joint or bursa injection/aspiration,1552,1506,225715.38,145.44,0.95
8,17000,Destruction of first premalignant skin lesion,1525,1464,190374.58,124.84,0.80
9,20552,"Trigger point injection, one or two muscles",1508,1454,172152.55,114.16,0.73


In [ ]:
# Calculate cumulative outpatient cost share to measure how concentrated spending is among the highest cost procedures.

procedure_concentration = con.execute("""
WITH procedure_cost AS (
    SELECT
        procedure_code,
        procedure_description,
        COUNT(*) AS claim_lines,
        SUM(allowed_amount) AS total_allowed_amount
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
    GROUP BY
        procedure_code,
        procedure_description
),

ranked_procedures AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY total_allowed_amount DESC
        ) AS cost_rank,
        100.0 * total_allowed_amount
            / SUM(total_allowed_amount) OVER () AS pct_outpatient_cost
    FROM procedure_cost
)

SELECT
    cost_rank,
    procedure_code,
    procedure_description,
    claim_lines,
    ROUND(total_allowed_amount, 2) AS total_allowed_amount,
    ROUND(pct_outpatient_cost, 2) AS pct_outpatient_cost,
    ROUND(
        SUM(pct_outpatient_cost) OVER (
            ORDER BY cost_rank
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS cumulative_pct_outpatient_cost
FROM ranked_procedures
ORDER BY cost_rank
""").df()

procedure_concentration

,cost_rank,procedure_code,procedure_description,claim_lines,total_allowed_amount,pct_outpatient_cost,cumulative_pct_outpatient_cost
0,1,47562,Laparoscopic cholecystectomy,1418,7060138.20,29.82,29.82
1,2,29881,Knee arthroscopy with meniscectomy,1456,4853707.90,20.50,50.31
2,3,49591,"Repair of anterior abdominal hernia, initial, ...",1538,4799370.60,20.27,70.58
3,4,66984,Cataract removal with intraocular lens insertion,1463,3643712.20,15.39,85.97
4,5,45378,Diagnostic colonoscopy,1529,1351430.49,5.71,91.68
5,6,43235,Diagnostic upper gastrointestinal endoscopy,1548,1128174.00,4.76,96.44
6,7,11102,"Tangential skin biopsy, single lesion",1526,254062.20,1.07,97.52
7,8,20610,Major joint or bursa injection/aspiration,1552,225715.38,0.95,98.47
8,9,17000,Destruction of first premalignant skin lesion,1525,190374.58,0.80,99.27
9,10,20552,"Trigger point injection, one or two muscles",1508,172152.55,0.73,100.00


In [ ]:
# Measure allowed amount variation within each outpatient procedure to identify procedures with meaningful cost differences.

procedure_cost_variation = con.execute("""
SELECT
    procedure_code,
    procedure_description,
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT provider_id) AS providers,
    ROUND(AVG(allowed_amount), 2) AS avg_allowed,
    ROUND(MEDIAN(allowed_amount), 2) AS median_allowed,
    ROUND(MIN(allowed_amount), 2) AS min_allowed,
    ROUND(MAX(allowed_amount), 2) AS max_allowed,
    ROUND(STDDEV_SAMP(allowed_amount), 2) AS std_dev_allowed
FROM claims
WHERE service_category = 'Outpatient Procedure'
GROUP BY
    procedure_code,
    procedure_description
ORDER BY SUM(allowed_amount) DESC
""").df()

procedure_cost_variation

,procedure_code,procedure_description,claim_lines,providers,avg_allowed,median_allowed,min_allowed,max_allowed,std_dev_allowed
0,47562,Laparoscopic cholecystectomy,1418,960,4978.94,4885.00,3840.0,6912.00,556.93
1,29881,Knee arthroscopy with meniscectomy,1456,952,3333.59,3279.70,2560.0,4608.00,375.80
2,49591,"Repair of anterior abdominal hernia, initial, ...",1538,1007,3120.53,3073.30,2400.0,4320.00,343.20
3,66984,Cataract removal with intraocular lens insertion,1463,962,2490.58,2451.71,1920.0,3408.43,268.15
4,45378,Diagnostic colonoscopy,1529,1010,883.87,872.93,680.0,1224.00,98.41
5,43235,Diagnostic upper gastrointestinal endoscopy,1548,1013,728.79,714.24,560.0,1008.00,81.30
6,11102,"Tangential skin biopsy, single lesion",1526,1003,166.49,164.01,128.0,230.40,18.64
7,20610,Major joint or bursa injection/aspiration,1552,1002,145.44,143.00,112.0,201.60,15.86
8,17000,Destruction of first premalignant skin lesion,1525,996,124.84,122.84,96.0,172.80,14.10
9,20552,"Trigger point injection, one or two muscles",1508,987,114.16,112.27,88.0,158.40,12.51


In [ ]:
# Compare provider level cost and volume for the four procedures responsible for most outpatient spending.

provider_procedure_summary = con.execute("""
SELECT
    provider_id,
    procedure_code,
    procedure_description,
    COUNT(*) AS claim_lines,
    ROUND(SUM(allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(MEDIAN(allowed_amount), 2) AS median_allowed_per_line
FROM claims
WHERE procedure_code IN ('47562', '29881', '49591', '66984')
GROUP BY
    provider_id,
    procedure_code,
    procedure_description
ORDER BY
    procedure_code,
    avg_allowed_per_line DESC
""").df()

provider_procedure_summary

,provider_id,procedure_code,procedure_description,claim_lines,total_allowed_amount,avg_allowed_per_line,median_allowed_per_line
0,PRV001226,29881,Knee arthroscopy with meniscectomy,1,4608.00,4608.00,4608.00
1,PRV002482,29881,Knee arthroscopy with meniscectomy,1,4544.05,4544.05,4544.05
2,PRV000285,29881,Knee arthroscopy with meniscectomy,1,4532.94,4532.94,4532.94
3,PRV000491,29881,Knee arthroscopy with meniscectomy,2,8923.02,4461.51,4461.51
4,PRV001457,29881,Knee arthroscopy with meniscectomy,1,4448.49,4448.49,4448.49
...,...,...,...,...,...,...,...
3876,PRV000507,66984,Cataract removal with intraocular lens insertion,1,1950.73,1950.73,1950.73
3877,PRV002227,66984,Cataract removal with intraocular lens insertion,1,1947.93,1947.93,1947.93
3878,PRV001525,66984,Cataract removal with intraocular lens insertion,1,1922.39,1922.39,1922.39
3879,PRV002253,66984,Cataract removal with intraocular lens insertion,1,1920.00,1920.00,1920.00


In [ ]:
# Examine how many claims each provider has for the top four procedures to determine whether provider averages are based on sufficient volume.

provider_volume_distribution = con.execute("""
WITH provider_volume AS (
    SELECT
        procedure_code,
        provider_id,
        COUNT(*) AS claim_lines
    FROM claims
    WHERE procedure_code IN ('47562', '29881', '49591', '66984')
    GROUP BY
        procedure_code,
        provider_id
)

SELECT
    procedure_code,
    COUNT(*) AS providers,
    ROUND(AVG(claim_lines), 2) AS avg_lines_per_provider,
    MEDIAN(claim_lines) AS median_lines_per_provider,
    MAX(claim_lines) AS max_lines_per_provider
FROM provider_volume
GROUP BY procedure_code
ORDER BY procedure_code
""").df()

provider_volume_distribution

,procedure_code,providers,avg_lines_per_provider,median_lines_per_provider,max_lines_per_provider
0,29881,952,1.53,1.0,7
1,47562,960,1.48,1.0,6
2,49591,1007,1.53,1.0,5
3,66984,962,1.52,1.0,6


In [ ]:
# Review the cleaned provider dataset schema before using provider attributes in the cost driver analysis.
display(providers.info())

# List the exact provider field names available for the next provider level analysis.
display(providers.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   provider_id     2500 non-null   object
 1   provider_name   2500 non-null   object
 2   provider_type   2500 non-null   object
 3   specialty       2500 non-null   object
 4   state           2500 non-null   object
 5   network_status  2500 non-null   object
dtypes: object(6)
memory usage: 117.3+ KB


None

['provider_id',
 'provider_name',
 'provider_type',
 'specialty',
 'state',
 'network_status']

In [ ]:
# Compare cost and utilization for the top four outpatient procedures by provider network status.

network_cost_summary = con.execute("""
SELECT
    c.procedure_code,
    c.procedure_description,
    p.network_status,
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT c.provider_id) AS providers,
    ROUND(SUM(c.allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(c.allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(MEDIAN(c.allowed_amount), 2) AS median_allowed_per_line
FROM claims c
INNER JOIN providers p
    ON c.provider_id = p.provider_id
WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
GROUP BY
    c.procedure_code,
    c.procedure_description,
    p.network_status
ORDER BY
    c.procedure_code,
    p.network_status
""").df()

network_cost_summary

,procedure_code,procedure_description,network_status,claim_lines,providers,total_allowed_amount,avg_allowed_per_line,median_allowed_per_line
0,29881,Knee arthroscopy with meniscectomy,In-Network,1141,751,3646386.12,3195.78,3194.05
1,29881,Knee arthroscopy with meniscectomy,Out-of-Network,315,201,1207321.78,3832.77,3814.24
2,47562,Laparoscopic cholecystectomy,In-Network,1140,775,5458945.64,4788.55,4774.91
3,47562,Laparoscopic cholecystectomy,Out-of-Network,278,185,1601192.56,5759.69,5781.41
4,49591,"Repair of anterior abdominal hernia, initial, ...",In-Network,1243,825,3735744.16,3005.43,3013.01
5,49591,"Repair of anterior abdominal hernia, initial, ...",Out-of-Network,295,182,1063626.44,3605.51,3625.71
6,66984,Cataract removal with intraocular lens insertion,In-Network,1169,778,2799974.42,2395.19,2397.27
7,66984,Cataract removal with intraocular lens insertion,Out-of-Network,294,184,843737.78,2869.86,2875.40


In [ ]:
# Calculate the share of top four outpatient procedure utilization and spending that occurs out of network.

network_share = con.execute("""
SELECT
    p.network_status,
    COUNT(*) AS claim_lines,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_claim_lines,
    ROUND(SUM(c.allowed_amount), 2) AS total_allowed_amount,
    ROUND(
        100.0 * SUM(c.allowed_amount)
        / SUM(SUM(c.allowed_amount)) OVER (),
        2
    ) AS pct_allowed_cost
FROM claims c
INNER JOIN providers p
    ON c.provider_id = p.provider_id
WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
GROUP BY p.network_status
ORDER BY total_allowed_amount DESC
""").df()

network_share

,network_status,claim_lines,pct_claim_lines,total_allowed_amount,pct_allowed_cost
0,In-Network,4693,79.88,15641050.34,76.83
1,Out-of-Network,1182,20.12,4715878.56,23.17


In [ ]:
# Compare cost and utilization across provider types and specialties for the top four outpatient procedures.

provider_characteristics = con.execute("""
SELECT
    p.provider_type,
    p.specialty,
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT c.provider_id) AS providers,
    ROUND(SUM(c.allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(c.allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(MEDIAN(c.allowed_amount), 2) AS median_allowed_per_line
FROM claims c
INNER JOIN providers p
    ON c.provider_id = p.provider_id
WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
GROUP BY
    p.provider_type,
    p.specialty
ORDER BY total_allowed_amount DESC
""").df()

provider_characteristics

,provider_type,specialty,claim_lines,providers,total_allowed_amount,avg_allowed_per_line,median_allowed_per_line
0,Hospital,General Acute Care,788,223,2741719.26,3479.34,3184.07
1,Physician,Dermatology,522,147,1860948.27,3565.04,3256.93
2,Physician,Primary Care,516,141,1790835.16,3470.61,3155.39
3,Physician,Pulmonology,510,140,1756972.06,3445.04,3172.50
4,Physician,Cardiology,487,138,1694241.62,3478.94,3159.50
5,Physician,Pediatrics,488,133,1673202.58,3428.69,3136.42
6,Physician,Orthopedics,473,126,1622430.77,3430.09,3127.99
7,Physician,Gastroenterology,432,117,1524819.38,3529.67,3185.90
8,Physician,Endocrinology,437,115,1505496.15,3445.07,3153.95
9,Physician,OB/GYN,430,117,1480893.50,3443.94,3180.35


In [ ]:
# Compare outpatient procedure spending and average allowed cost across provider states to identify geographic cost variation.

state_cost_summary = con.execute("""
SELECT
    p.state,
    COUNT(*) AS claim_lines,
    COUNT(DISTINCT c.provider_id) AS providers,
    ROUND(SUM(c.allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(c.allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(MEDIAN(c.allowed_amount), 2) AS median_allowed_per_line
FROM claims c
INNER JOIN providers p
    ON c.provider_id = p.provider_id
WHERE c.service_category = 'Outpatient Procedure'
GROUP BY p.state
ORDER BY total_allowed_amount DESC
""").df()

state_cost_summary

,state,claim_lines,providers,total_allowed_amount,avg_allowed_per_line,median_allowed_per_line
0,CA,1844,202,2949233.00,1599.37,814.26
1,TX,1281,152,2079332.68,1623.21,821.67
2,FL,1012,115,1683038.38,1663.08,851.18
3,NY,999,118,1592963.60,1594.56,807.22
4,IL,547,44,877046.55,1603.38,759.26
5,PA,552,62,870601.01,1577.18,828.02
6,OH,538,73,831016.12,1544.64,767.89
7,NC,477,62,778284.31,1631.62,824.06
8,MI,482,51,767338.03,1591.99,756.52
9,GA,481,71,739574.44,1537.58,760.77


In [ ]:
# Compare average allowed amounts by state while controlling for procedure and network status among the top four outpatient procedures.

state_adjusted_cost = con.execute("""
SELECT
    p.state,
    c.procedure_code,
    p.network_status,
    COUNT(*) AS claim_lines,
    ROUND(AVG(c.allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(MEDIAN(c.allowed_amount), 2) AS median_allowed_per_line
FROM claims c
INNER JOIN providers p
    ON c.provider_id = p.provider_id
WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
GROUP BY
    p.state,
    c.procedure_code,
    p.network_status
HAVING COUNT(*) >= 10
ORDER BY
    c.procedure_code,
    p.network_status,
    avg_allowed_per_line DESC
""").df()

state_adjusted_cost

,state,procedure_code,network_status,claim_lines,avg_allowed_per_line,median_allowed_per_line
0,CT,29881,In-Network,10,3284.23,3274.97
1,KS,29881,In-Network,13,3281.34,3324.14
2,GA,29881,In-Network,33,3272.98,3276.60
3,TN,29881,In-Network,10,3271.59,3205.92
4,AR,29881,In-Network,10,3257.95,3277.95
...,...,...,...,...,...,...
166,NJ,66984,Out-of-Network,13,2845.54,2793.07
167,WI,66984,Out-of-Network,12,2834.88,2726.61
168,PA,66984,Out-of-Network,13,2811.12,2794.59
169,FL,66984,Out-of-Network,23,2799.32,2794.95


In [ ]:
# Summarize state level cost variation within each procedure and network group to determine whether geography adds a meaningful cost effect.

geographic_variation = con.execute("""
WITH state_cost AS (
    SELECT
        p.state,
        c.procedure_code,
        p.network_status,
        COUNT(*) AS claim_lines,
        AVG(c.allowed_amount) AS avg_allowed_per_line
    FROM claims c
    INNER JOIN providers p
        ON c.provider_id = p.provider_id
    WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
    GROUP BY
        p.state,
        c.procedure_code,
        p.network_status
    HAVING COUNT(*) >= 10
)

SELECT
    procedure_code,
    network_status,
    COUNT(*) AS states,
    ROUND(MIN(avg_allowed_per_line), 2) AS lowest_state_avg,
    ROUND(MAX(avg_allowed_per_line), 2) AS highest_state_avg,
    ROUND(AVG(avg_allowed_per_line), 2) AS avg_across_states,
    ROUND(MAX(avg_allowed_per_line) - MIN(avg_allowed_per_line), 2)
        AS state_cost_range,
    ROUND(
        100.0 *
        (MAX(avg_allowed_per_line) - MIN(avg_allowed_per_line))
        / AVG(avg_allowed_per_line),
        2
    ) AS range_pct_of_avg
FROM state_cost
GROUP BY
    procedure_code,
    network_status
ORDER BY
    procedure_code,
    network_status
""").df()

geographic_variation


,procedure_code,network_status,states,lowest_state_avg,highest_state_avg,avg_across_states,state_cost_range,range_pct_of_avg
0,29881,In-Network,35,3087.30,3284.23,3194.72,196.93,6.16
1,29881,Out-of-Network,11,3683.67,3876.48,3798.50,192.80,5.08
2,47562,In-Network,34,4539.02,5016.80,4791.66,477.78,9.97
3,47562,Out-of-Network,9,5524.50,5934.70,5761.46,410.20,7.12
4,49591,In-Network,32,2902.31,3113.56,3004.00,211.24,7.03
5,49591,Out-of-Network,6,3522.66,3797.53,3627.48,274.87,7.58
6,66984,In-Network,32,2345.99,2473.70,2399.06,127.71,5.32
7,66984,Out-of-Network,12,2761.39,2929.79,2854.53,168.41,5.90


In [ ]:
# Measure how concentrated outpatient procedure spending is across members by grouping members into cost deciles.

member_cost_concentration = con.execute("""
WITH member_cost AS (
    SELECT
        member_id,
        COUNT(*) AS claim_lines,
        SUM(allowed_amount) AS total_allowed_amount
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
    GROUP BY member_id
),

ranked_members AS (
    SELECT
        *,
        NTILE(10) OVER (
            ORDER BY total_allowed_amount DESC
        ) AS cost_decile
    FROM member_cost
)

SELECT
    cost_decile,
    COUNT(*) AS members,
    SUM(claim_lines) AS claim_lines,
    ROUND(SUM(total_allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(total_allowed_amount), 2) AS avg_allowed_per_member,
    ROUND(
        100.0 * SUM(total_allowed_amount)
        / SUM(SUM(total_allowed_amount)) OVER (),
        2
    ) AS pct_outpatient_cost
FROM ranked_members
GROUP BY cost_decile
ORDER BY cost_decile
""").df()

member_cost_concentration

,cost_decile,members,claim_lines,total_allowed_amount,avg_allowed_per_member,pct_outpatient_cost
0,1,1094,2600.0,7619421.05,6964.74,32.18
1,2,1094,1716.0,4881537.33,4462.10,20.62
2,3,1093,1579.0,3697845.80,3383.21,15.62
3,4,1093,1354.0,3188157.91,2916.89,13.46
4,5,1093,1622.0,2244863.30,2053.85,9.48
5,6,1093,1344.0,957381.93,875.92,4.04
6,7,1093,1497.0,631405.52,577.68,2.67
7,8,1093,1165.0,187497.69,171.54,0.79
8,9,1093,1093.0,148690.02,136.04,0.63
9,10,1093,1093.0,122037.55,111.65,0.52


In [ ]:
# Compare procedure mix across member cost deciles to determine whether high cost members receive more expensive types of outpatient procedures.

member_decile_procedures = con.execute("""
WITH member_cost AS (
    SELECT
        member_id,
        SUM(allowed_amount) AS total_allowed_amount
    FROM claims
    WHERE service_category = 'Outpatient Procedure'
    GROUP BY member_id
),

ranked_members AS (
    SELECT
        member_id,
        NTILE(10) OVER (
            ORDER BY total_allowed_amount DESC
        ) AS cost_decile
    FROM member_cost
)

SELECT
    r.cost_decile,
    c.procedure_code,
    c.procedure_description,
    COUNT(*) AS claim_lines,
    ROUND(SUM(c.allowed_amount), 2) AS total_allowed_amount,
    ROUND(AVG(c.allowed_amount), 2) AS avg_allowed_per_line,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY r.cost_decile),
        2
    ) AS pct_decile_claim_lines
FROM claims c
INNER JOIN ranked_members r
    ON c.member_id = r.member_id
WHERE c.service_category = 'Outpatient Procedure'
GROUP BY
    r.cost_decile,
    c.procedure_code,
    c.procedure_description
ORDER BY
    r.cost_decile,
    total_allowed_amount DESC
""").df()

member_decile_procedures

,cost_decile,procedure_code,procedure_description,claim_lines,total_allowed_amount,avg_allowed_per_line,pct_decile_claim_lines
0,1,47562,Laparoscopic cholecystectomy,805,4209773.81,5229.53,30.96
1,1,29881,Knee arthroscopy with meniscectomy,374,1255448.77,3356.81,14.38
2,1,49591,"Repair of anterior abdominal hernia, initial, ...",365,1141763.06,3128.12,14.04
3,1,66984,Cataract removal with intraocular lens insertion,284,712259.10,2507.95,10.92
4,1,45378,Diagnostic colonoscopy,147,130104.75,885.07,5.65
...,...,...,...,...,...,...,...
65,9,11102,"Tangential skin biopsy, single lesion",135,19140.40,141.78,12.35
66,9,20552,"Trigger point injection, one or two muscles",138,18306.39,132.66,12.63
67,10,20552,"Trigger point injection, one or two muscles",641,70024.81,109.24,58.65
68,10,17000,Destruction of first premalignant skin lesion,405,46376.84,114.51,37.05


In [ ]:
# Estimate the potential cost opportunity by comparing out-of-network costs with observed in-network averages for the same procedure.

network_opportunity = con.execute("""
WITH procedure_network AS (
    SELECT
        c.procedure_code,
        p.network_status,
        COUNT(*) AS claim_lines,
        AVG(c.allowed_amount) AS avg_allowed_per_line,
        SUM(c.allowed_amount) AS total_allowed_amount
    FROM claims c
    INNER JOIN providers p
        ON c.provider_id = p.provider_id
    WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
    GROUP BY
        c.procedure_code,
        p.network_status
),

comparison AS (
    SELECT
        o.procedure_code,
        o.claim_lines AS out_of_network_lines,
        o.avg_allowed_per_line AS out_of_network_avg,
        i.avg_allowed_per_line AS in_network_avg,
        o.total_allowed_amount AS out_of_network_cost
    FROM procedure_network o
    INNER JOIN procedure_network i
        ON o.procedure_code = i.procedure_code
    WHERE o.network_status = 'Out-of-Network'
      AND i.network_status = 'In-Network'
)

SELECT
    procedure_code,
    out_of_network_lines,
    ROUND(out_of_network_avg, 2) AS out_of_network_avg,
    ROUND(in_network_avg, 2) AS in_network_avg,
    ROUND(out_of_network_avg - in_network_avg, 2) AS avg_cost_difference,
    ROUND(out_of_network_cost, 2) AS out_of_network_cost,
    ROUND(
        out_of_network_lines *
        (out_of_network_avg - in_network_avg),
        2
    ) AS estimated_cost_opportunity
FROM comparison
ORDER BY estimated_cost_opportunity DESC
""").df()

network_opportunity

,procedure_code,out_of_network_lines,out_of_network_avg,in_network_avg,avg_cost_difference,out_of_network_cost,estimated_cost_opportunity
0,47562,278,5759.69,4788.55,971.14,1601192.56,269975.99
1,29881,315,3832.77,3195.78,636.99,1207321.78,200650.77
2,49591,295,3605.51,3005.43,600.09,1063626.44,177025.85
3,66984,294,2869.86,2395.19,474.67,843737.78,139552.60


In [ ]:
# Model estimated savings if 25%, 50%, 75%, or 100% of current OON utilization shifted to in-network cost levels.

network_scenarios = con.execute("""
WITH procedure_network AS (
    SELECT
        c.procedure_code,
        p.network_status,
        COUNT(*) AS claim_lines,
        AVG(c.allowed_amount) AS avg_allowed_per_line
    FROM claims c
    INNER JOIN providers p
        ON c.provider_id = p.provider_id
    WHERE c.procedure_code IN ('47562', '29881', '49591', '66984')
    GROUP BY
        c.procedure_code,
        p.network_status
),

opportunity AS (
    SELECT
        SUM(
            o.claim_lines *
            (o.avg_allowed_per_line - i.avg_allowed_per_line)
        ) AS full_opportunity
    FROM procedure_network o
    INNER JOIN procedure_network i
        ON o.procedure_code = i.procedure_code
    WHERE o.network_status = 'Out-of-Network'
      AND i.network_status = 'In-Network'
)

SELECT
    scenario,
    ROUND(full_opportunity * shift_rate, 2) AS estimated_savings
FROM opportunity
CROSS JOIN (
    VALUES
        ('25% OON shift', 0.25),
        ('50% OON shift', 0.50),
        ('75% OON shift', 0.75),
        ('100% OON shift', 1.00)
) AS scenarios(scenario, shift_rate)
ORDER BY shift_rate
""").df()

network_scenarios

,scenario,estimated_savings
0,25% OON shift,196801.30
1,50% OON shift,393602.60
2,75% OON shift,590403.91
3,100% OON shift,787205.21
